<a href="https://colab.research.google.com/github/evlannikovoleg-prog/ML-Prof/blob/main/hw8/hw8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Проверка трансформеров в бою

### Часть 0. Подготовка окружения и скачивание данных

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Читаем файлы, которые вы вручную перетащили в левую панель Colab
train_df = pd.read_csv("in_domain_train.csv")
test_df = pd.read_csv("in_domain_dev.csv") # dev по заданию используем как тест

# 2. Разбиваем in_domain_train на train (85%) и val (15%) с сохранением баланса классов (stratify)
train_data, val_data = train_test_split(
    train_df,
    test_size=0.15,
    random_state=42,
    stratify=train_df['acceptable']
)

print("Файлы RuCoLA успешно прочитаны локально!")
print(f"Размер обучающей выборки (train): {len(train_data)}")
print(f"Размер валидационной выборки (val): {len(val_data)}")
print(f"Размер тестовой выборки (test/dev): {len(test_df)}")

# Выводим кусочек данных для проверки
train_data.head(3)

Файлы RuCoLA успешно прочитаны локально!
Размер обучающей выборки (train): 6688
Размер валидационной выборки (val): 1181
Размер тестовой выборки (test/dev): 983


,id,sentence,acceptable,error_type,detailed_source
7444,7444,Ребята накупили много хороших книг.,1,0,Testelets
358,358,"Иди, а то Иван скажет, что ты заставил его ждать.",1,0,Paducheva2010
5511,5511,Египетский мост в Санкт-Петербурге известен бл...,0,Syntax,USE8


### Часть 1. Fine-tuning энкодера (ruBERT / ruRoBERTa)

In [6]:
# --- 1. Установка библиотек Hugging Face ---
!pip install -q transformers[torch] accelerate datasets

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset as HFDataset
from sklearn.metrics import accuracy_score, matthews_corrcoef

# Проверяем, что Colab выделил нам T4 GPU (должно быть 'cuda')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Работаем на устройстве: {device}")


Работаем на устройстве: cuda


In [7]:
# --- 2. Инициализация токенизатора ruBERT ---
model_name = "ai-forever/ruBert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Преобразуем наши Pandas DataFrame из прошлого шага в формат Hugging Face Dataset
# Переименовываем колонку 'acceptable' в 'label', как требует стандарт Trainer
train_dataset = HFDataset.from_pandas(train_data[['sentence', 'acceptable']].rename(columns={'acceptable': 'label'}))
val_dataset = HFDataset.from_pandas(val_data[['sentence', 'acceptable']].rename(columns={'acceptable': 'label'}))
test_dataset = HFDataset.from_pandas(test_df[['sentence', 'acceptable']].rename(columns={'acceptable': 'label'}))


config.json:   0%|          | 0.00/590 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.78M [00:00<?, ?B/s]

In [8]:
# --- 3. Токенизация текстов ---
def tokenize_function(examples):
    # Ограничим длину предложения 64 токенами (для RuCoLA этого более чем достаточно)
    return tokenizer(examples['sentence'], truncation=True, padding='max_length', max_length=64)

print("Токенизируем текстовые выборки...")
train_tokenized = train_dataset.map(tokenize_function, batched=True)
val_tokenized = val_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)



Токенизируем текстовые выборки...


Map:   0%|          | 0/6688 [00:00<?, ? examples/s]

Map:   0%|          | 0/1181 [00:00<?, ? examples/s]

Map:   0%|          | 0/983 [00:00<?, ? examples/s]

In [9]:
# --- 4. Кастомная функция расчета метрик (Accuracy и MCC) ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    mcc = matthews_corrcoef(labels, predictions) # Ключевая метрика бенчмарка RuCoLA
    return {"accuracy": acc, "mcc": mcc}


In [10]:
# --- 5. Загрузка предобученной модели ruBERT ---
# Указываем num_labels=2 для задачи бинарной классификации (приемлемо / неприемлемо)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  716MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: ai-forever/ruBert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params we

In [12]:
# --- 6. Настройка параметров обучения под T4 GPU (Исправленная версия) ---
training_args = TrainingArguments(
    output_dir="./results_rubert",
    learning_rate=2e-5,               # Стандартный аккуратный шаг обучения для BERT
    per_device_train_batch_size=32,    # Оптимальный размер батча для 16 ГБ видеопамяти T4
    per_device_eval_batch_size=32,
    num_train_epochs=3,                # 3 эпохи достаточно, чтобы BERT уловил синтаксические ошибки
    weight_decay=0.01,
    eval_strategy="epoch",             # ИСПРАВЛЕНО: используем eval_strategy вместо evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,       # В конце обучения загружаем веса с лучшим Val MCC
    metric_for_best_model="mcc",
    fp16=True,                         # Включаем Mixed Precision (float16) — это ускорит T4 в 2-3 раза!
    logging_steps=50,
    report_to="none"                   # Отключаем логирование в сторонние сервисы (wandb)
)

In [13]:
# --- 7. Запуск Fine-tuning с помощью Trainer ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
)

print("\nЗапуск Fine-tuning ruBERT на GPU...")
trainer.train()




Запуск Fine-tuning ruBERT на GPU...


Epoch,Training Loss,Validation Loss,Accuracy,Mcc
1,0.486843,0.523566,0.780694,0.315945
2,0.342133,0.498858,0.804403,0.417874
3,0.234098,0.578102,0.812024,0.448542


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=627, training_loss=0.37415206565431053, metrics={'train_runtime': 116.4176, 'train_samples_per_second': 172.345, 'train_steps_per_second': 5.386, 'total_flos': 659882526842880.0, 'train_loss': 0.37415206565431053, 'epoch': 3.0})

In [14]:
# --- 8. Финальная оценка качества модели на тестовой (dev) выборке ---
print("\nТестирование лучшей модели на отложенной Dev-выборке:")
test_results = trainer.evaluate(test_tokenized)

# Сохраняем финальные метрики в переменные для итоговой сравнительной таблицы задачи
rubert_acc = test_results['eval_accuracy']
rubert_mcc = test_results['eval_mcc']

print(f"\nФинальные результаты ruBERT-base на RuCoLA:")
print(f"Accuracy: {rubert_acc:.4f}")
print(f"MCC Score: {rubert_mcc:.4f}")


Тестирование лучшей модели на отложенной Dev-выборке:


Training Loss,Validation Loss,Epoch,Accuracy,Mcc
0.234098,0.608559,3,0.793489,0.381844



Финальные результаты ruBERT-base на RuCoLA:
Accuracy: 0.7935
MCC Score: 0.3818


### Часть 2. Zero-Shot / Few-Shot методы на RuGPT-3 (Этапы А и Б)

In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm

# 1. Загрузка генеративной модели RuGPT-3 Small и токенизатора
gpt_model_name = "ai-forever/rugpt3small_based_on_gpt2"
gpt_tokenizer = AutoTokenizer.from_pretrained(gpt_model_name)
gpt_model = AutoModelForCausalLM.from_pretrained(gpt_model_name).to(device)
gpt_model.eval()

# Получаем ID токенов для ответов "Да" и "Нет" (с учетом пробела перед словом, как генерирует GPT)
token_yes_id = gpt_tokenizer.encode(" Да")[0]
token_no_id = gpt_tokenizer.encode(" Нет")[0]

config.json:   0%|          | 0.00/720 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.71M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.27M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/574 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  551MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3small_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
# 2. Определение двух вариантов затравок (Промптов)
prompts_variants = {
    "Вариант_А": {
        "instruction": "Определи, корректно ли написано предложение с точки зрения русского языка. Ответь только 'Да' или 'Нет'.\n",
        "template": "Предложение: \"{sentence}\"\nКорректно: {label}"
    },
    "Вариант_Б": {
        "instruction": "Твоя задача — найти грамматические и синтаксические ошибки в тексте. Если ошибок нет, напиши 'Да', если есть ошибки — 'Нет'.\n",
        "template": "Текст: {sentence}\nПравильно: {label}"
    }
}

In [17]:
# 3. Функция для сборки Few-Shot контекста из обучающей выборки (train_data)
def get_few_shot_context(train_df, num_shots, prompt_style):
    if num_shots == 0:
        return prompt_style["instruction"]

    # Берем случайные сбалансированные примеры из train_data
    sampled_correct = train_df[train_df['acceptable'] == 1].sample(num_shots // 2 + num_shots % 2, random_state=42)
    sampled_incorrect = train_df[train_df['acceptable'] == 0].sample(num_shots // 2, random_state=42)
    few_shot_samples = pd.concat([sampled_correct, sampled_incorrect]).sample(frac=1, random_state=42) # перемешиваем

    context = prompt_style["instruction"]
    for _, row in few_shot_samples.iterrows():
        ans = "Да" if row['acceptable'] == 1 else "Нет"
        context += prompt_style["template"].format(sentence=row['sentence'], label=ans) + "\n\n"
    return context

In [18]:
# 4. Сетка экспериментов
shots_variants = [0, 1, 2, 4]
gpt_results_log = []

print("Запуск Zero-Shot и Few-Shot экспериментов на RuGPT-3...")

for p_name, p_style in prompts_variants.items():
    for num_shots in shots_variants:
        preds = []
        labels = []

        # Строим базовый Few-Shot контекст (инструкция + примеры)
        base_context = get_few_shot_context(train_data, num_shots, p_style)

        # Проверяем модель на тестовой (dev) выборке test_df
        for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc=f"{p_name} | {num_shots}-Shot", leave=False):
            # Дописываем текущее тестируемое предложение, оставляя поле {label} пустым для прогноза модели
            full_prompt = base_context + p_style["template"].format(sentence=row['sentence'], label="")

            # Токенизируем промпт
            inputs = gpt_tokenizer(full_prompt, return_tensors="pt").to(device)

            with torch.no_grad():
                outputs = gpt_model(**inputs)
                # Берем логиты (предсказания) самого последнего токена в последовательности
                next_token_logits = outputs.logits[0, -1, :]

                # Извлекаем логиты только для интересующих нас токенов "Да" и "Нет"
                logit_yes = next_token_logits[token_yes_id].item()
                logit_no = next_token_logits[token_no_id].item()

            # Классифицируем по максимальному логиту
            pred_class = 1 if logit_yes > logit_no else 0
            preds.append(pred_class)
            labels.append(row['acceptable'])

        # Считаем метрики качества для данной комбинации
        acc = accuracy_score(labels, preds)
        mcc = matthews_corrcoef(labels, preds)

        gpt_results_log.append({
            "Prompt": p_name,
            "Shots": num_shots,
            "Accuracy": acc,
            "MCC": mcc
        })
        print(f"{p_name} | {num_shots}-Shot -> Accuracy: {acc:.4f} | MCC Score: {mcc:.4f}")

# Превращаем результаты в удобный DataFrame
gpt_results_df = pd.DataFrame(gpt_results_log)

Запуск Zero-Shot и Few-Shot экспериментов на RuGPT-3...


Вариант_А | 0-Shot:   0%|          | 0/983 [00:00<?, ?it/s]

Вариант_А | 0-Shot -> Accuracy: 0.7447 | MCC Score: -0.0186


Вариант_А | 1-Shot:   0%|          | 0/983 [00:00<?, ?it/s]

Вариант_А | 1-Shot -> Accuracy: 0.7447 | MCC Score: -0.0186


...

Вариант_Б | 1-Shot -> Accuracy: 0.7447 | MCC Score: -0.0186


Вариант_Б | 2-Shot:   0%|          | 0/983 [00:00<?, ?it/s]

Вариант_Б | 2-Shot -> Accuracy: 0.4547 | MCC Score: 0.0598


Вариант_Б | 4-Shot:   0%|          | 0/983 [00:00<?, ?it/s]

Вариант_Б | 4-Shot -> Accuracy: 0.3225 | MCC Score: -0.0140


### Часть 3. Fine-tuning модели RuT5 (text-to-text)

In [19]:
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
# 1. Инициализируем токенизатор и саму модель RuT5-base
t5_model_name = "sberbank-ai/ruT5-base"
t5_tokenizer = T5Tokenizer.from_pretrained(t5_model_name)
t5_model = T5ForConditionalGeneration.from_pretrained(t5_model_name).to(device)

# Целевые текстовые метки для text-to-text задачи
POS_LABEL = "yes"
NEG_LABEL = "no"

tokenizer_config.json:   0%|          | 0.00/20.4k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B / 1.00MB            

spiece.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  892MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

In [21]:
# 2. Переводим наши Pandas Dataframe в формат Hugging Face Dataset
train_dataset_t5 = HFDataset.from_pandas(train_data[['sentence', 'acceptable']])
val_dataset_t5 = HFDataset.from_pandas(val_data[['sentence', 'acceptable']])
test_dataset_t5 = HFDataset.from_pandas(test_df[['sentence', 'acceptable']])

# 3. Функция предобработки (токенизация входа и таргета)
def preprocess_t5_function(examples):
    # Переводим метки 1/0 в текст "yes"/"no"
    labels_text = [POS_LABEL if label == 1 else NEG_LABEL for label in examples["acceptable"]]

    # ИСПРАВЛЕНО: Токенизируем входные предложения и текстовые ответы ОДНИМ вызовом,
    # используя официальный параметр text_target вместо устаревшего as_target_tokenizer()
    model_inputs = t5_tokenizer(
        examples["sentence"],
        text_target=labels_text,
        max_length=64,
        truncation=True,
        padding=False
    )

    return model_inputs

print("Токенизируем данные под формат Text-to-Text...")
train_tokenized_t5 = train_dataset_t5.map(preprocess_t5_function, batched=True, remove_columns=["sentence", "acceptable"])
val_tokenized_t5 = val_dataset_t5.map(preprocess_t5_function, batched=True, remove_columns=["sentence", "acceptable"])
test_tokenized_t5 = test_dataset_t5.map(preprocess_t5_function, batched=True, remove_columns=["sentence", "acceptable"])


Токенизируем данные под формат Text-to-Text...


Map:   0%|          | 0/6688 [00:00<?, ? examples/s]

Map:   0%|          | 0/1181 [00:00<?, ? examples/s]

Map:   0%|          | 0/983 [00:00<?, ? examples/s]

In [22]:
# 4. Кастомная функция расчета метрик (Раскодирует текст обратно в числа 1 и 0)
def compute_t5_metrics(eval_pred):
    predictions, labels = eval_pred

    # Декодируем сгенерированные моделью токены обратно в текст
    string_preds = t5_tokenizer.batch_decode(predictions, skip_special_tokens=True)
    int_preds = [1 if pred.strip().lower() == POS_LABEL else 0 for pred in string_preds]

    # Заменяем технические маски -100 (если есть) на токен пада для корректного декодирования
    labels = np.where(labels != -100, labels, t5_tokenizer.pad_token_id)
    string_labels = t5_tokenizer.batch_decode(labels, skip_special_tokens=True)
    int_labels = [1 if label.strip().lower() == POS_LABEL else 0 for label in string_labels]

    acc = accuracy_score(int_labels, int_preds)
    mcc = matthews_corrcoef(int_labels, int_preds)
    return {"accuracy": acc, "mcc": mcc}

# Сборщик батчей для Seq2Seq моделей
data_collator = DataCollatorForSeq2Seq(t5_tokenizer, model=t5_model, pad_to_multiple_of=8)


In [23]:
# 5. Настройка параметров обучения под T4 GPU
t5_training_args = Seq2SeqTrainingArguments(
    output_dir="./results_rut5",
    learning_rate=5e-4,                # Для T5 шаг обучения обычно берут чуть больше, чем для BERT
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="mcc",
    fp16=True,                         # Ускорение на GPU
    predict_with_generate=True,        # Важно! Включает генерацию текста во время валидации
    generation_max_length=10,
    report_to="none"
)

In [25]:
# 6. Инициализация и запуск Seq2SeqTrainer
t5_trainer = Seq2SeqTrainer(
    model=t5_model,
    args=t5_training_args,
    train_dataset=train_tokenized_t5,
    eval_dataset=val_tokenized_t5,
    processing_class=t5_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_t5_metrics,
)

print("\nЗапуск Fine-tuning RuT5 на GPU...")
t5_trainer.train()



Запуск Fine-tuning RuT5 на GPU...


Epoch,Training Loss,Validation Loss,Accuracy,Mcc
1,No log,0.204052,0.745131,0.000000
2,No log,0.197706,0.755292,0.220384
3,0.378343,0.203231,0.751905,0.243893


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=627, training_loss=0.3385253310013426, metrics={'train_runtime': 237.0446, 'train_samples_per_second': 84.642, 'train_steps_per_second': 2.645, 'total_flos': 910087567441920.0, 'train_loss': 0.3385253310013426, 'epoch': 3.0})

In [26]:
# 7. Финальное тестирование на отложенной выборке (in_domain_dev)
print("\nТестирование лучшей модели RuT5 на отложенной Dev-выборке:")
t5_test_results = t5_trainer.evaluate(test_tokenized_t5, metric_key_prefix="test")

rut5_acc = t5_test_results['test_accuracy']
rut5_mcc = t5_test_results['test_mcc']

print(f"\nФинальные результаты RuT5 на RuCoLA:")
print(f"Accuracy: {rut5_acc:.4f}")
print(f"MCC Score: {rut5_mcc:.4f}")


Тестирование лучшей модели RuT5 на отложенной Dev-выборке:


Training Loss,Validation Loss,Epoch,Accuracy,Mcc
0.378343,0.203693,3,0.770092,0.304403



Финальные результаты RuT5 на RuCoLA:
Accuracy: 0.7701
MCC Score: 0.3044


### Часть 4. Сравнительный анализ результатов
Итоговая сравнительная таблица:

In [27]:
# Собираем все зафиксированные в ходе лабораторной работы метрики
final_results_data = {
    "Модель / Метод": [
        "ruBERT-base (Fine-tuning)",
        "RuT5-base (Fine-tuning)",
        "RuGPT3-Small (Prompt А, 0-Shot)",
        "RuGPT3-Small (Prompt А, 4-Shot)",
        "RuGPT3-Small (Prompt Б, 2-Shot)"
    ],
    "Accuracy": [
        0.7935,
        0.7701,
        0.7447,
        0.3123,
        0.4547
    ],
    "MCC Score": [
        0.3818,
        0.3044,
        -0.0186,
        -0.0285,
        0.0598
    ]
}

comparison_df = pd.DataFrame(final_results_data)

print("ИТОГОВАЯ ТАБЛИЦА СРАВНЕНИЯ МОДЕЛЕЙ НА BENCHMARK RuCoLA:")
comparison_df

ИТОГОВАЯ ТАБЛИЦА СРАВНЕНИЯ МОДЕЛЕЙ НА BENCHMARK RuCoLA:


,Модель / Метод,Accuracy,MCC Score
0,ruBERT-base (Fine-tuning),0.7935,0.3818
1,RuT5-base (Fine-tuning),0.7701,0.3044
2,"RuGPT3-Small (Prompt А, 0-Shot)",0.7447,-0.0186
3,"RuGPT3-Small (Prompt А, 4-Shot)",0.3123,-0.0285
4,"RuGPT3-Small (Prompt Б, 2-Shot)",0.4547,0.0598


В ходе лабораторной работы было проведено исследование эффективности различных архитектур трансформеров (ruBERT, RuT5, RuGPT-3) в задаче оценки лингвистической приемлемости (Linguistic Acceptability) на базе русскоязычного бенчмарка RuCoLA.
На основе полученной сравнительной таблицы можно сделать следующие ключевые выводы:
1. Безоговорочное превосходство энкодеров:
Модель ruBERT-base продемонстрировала наилучшее качество классификации, достигнув MCC Score: 0.3818 и Accuracy: 79.35%. Двунаправленная архитектура слоя внимания (bidirectional attention) позволяет энкодерам оценивать синтаксические и морфологические связи в предложении целиком, что делает их наиболее эффективным инструментом для поиска тонких языковых ошибок.
2. Специфика генеративного подхода в классификации:
Модель RuT5-base показала сопоставимую точность (Accuracy: 77.01%), но уступила по метрике MCC (0.3044). Поскольку задача для T5 формулировалась в парадигме Text-to-Text (генерация токенов "yes"/"no"), авторегрессионный декодер тратит часть емкости на выстраивание последовательности, что делает итоговый вердикт чуть менее чувствительным к скрытым грамматическим аномалиям по сравнению с чистой классификационной головой BERT.
3. Неэффективность In-Context Learning (Few-Shot) на малых масштабах:
Эксперименты с RuGPT-3 Small наглядно доказали, что генеративные модели малой емкости (125M параметров) не способны стабильно решать логические задачи «на лету».
	• В режимах 0-Shot и 1-Shot модель продемонстрировала классический коллапс частотности токенов (выдавала ответ «Да» на любой запрос из-за его популярности в языке), что привело к отрицательному значению MCC (~-0.01).
	• Незначительный прорыв удалось зафиксировать только в Варианте промпта Б при 2-Shot (MCC: 0.0598), где формулировка триггерных слов помогла модели частично уловить контекст. При увеличении числа примеров до 4 модель перегрузила контекстное окно и полностью потеряла нить инструкции.

## Итог:
Для решения узкоспециализированных лингвистических задач на русском языке полноценное дообучение (Fine-tuning) относительно небольших моделей-энкодеров (BERT/RoBERTa) является наиболее экономически и технически оправданным подходом, кратно превосходящим методы Zero/Few-Shot генерации.